# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. We will programmatically enumerate all record sets, then for each record set, list its fields and columns.

In [ ]:
record_sets = dataset.record_sets

if not record_sets:
    print('No record sets found in this dataset.')
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"Record set @id: {rs.id}\n  name: {rs.name}\n  description: {rs.description if hasattr(rs, 'description') else '-'}")
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields:")
            for field in rs.fields:
                print(f"    - Field @id: {field.id} | name: {getattr(field, 'name', '-')}")
        if hasattr(rs, 'columns') and rs.columns:
            print("  Columns:")
            for col in rs.columns:
                print(f"    - Column @id: {col.id} | name: {getattr(col, 'name', '-')}")
        print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Below, we fetch all available record sets (by their `@id`), and extract them into DataFrames, using only the `@id` to reference them.

In [ ]:
# Build a list of record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # create dataframe only if there are records
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set: {record_set_id} - DataFrame shape: {dataframes[record_set_id].shape}")
    else:
        print(f"Record set {record_set_id} has no records.")

# Show columns for the first available DataFrame
if dataframes:
    first_rs = next(iter(dataframes))
    print(f"\nColumns for record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping by attributes. 

Below, we select the first numeric field available for a record set, filter, normalize, and group by a categorical field if present. All references are by `@id`.

In [ ]:
import numpy as np

if dataframes:
    # Select a record set to analyze
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    print(f"Analyzing record set: {record_set_id}\n")

    # Try to heuristically pick the first numeric field for demonstration
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if not numeric_fields:
        # Try to convert columns to numeric if any
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except:
                continue
        numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")

        # Example threshold (using mean if possible, else 10)
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to pick a group field (categorical)
        group_fields = [col for col in df.columns if df[col].dtype == 'object' and df[col].nunique() < 20]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouping by {group_field}:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print('No suitable categorical field found for grouping.')
    else:
        print('No numeric field available for EDA.')
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if dataframes and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} in record set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Show boxplot by group field if available
    if group_fields:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_fields[0]], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_fields[0]}")
        plt.xlabel(group_fields[0])
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset on adoption predictors for indigenous and modern knowledge in rangeland management using the `mlcroissant` library. We demonstrated how to discover record sets, fields, and columns via their `@id`, programmatically load records, and perform simple exploratory data analysis and visualization. 

- Always reference dataset structures (record sets, fields, columns) by their unique `@id` field.
- The Croissant format enables robust, FAIR-compliant dataset discovery, flexible extraction, and direct integration with data science workflows.
- Further analyses can be performed by inspecting all field and column `@id`s and applying domain-specific processing or modeling.